# AI Traffic Violation Detection - Colab demo

Detects vehicles crossing a stop line on **red**, saves evidence, and shows the results. Run the cells top to bottom.

1. **Runtime -> Change runtime type -> GPU** (T4 is fine).
2. Upload the project (zip) and a traffic video.
3. Define the stop line and signal ROI (headless, there is no window in Colab).
4. Run the pipeline, review events and evidence, evaluate, benchmark, open the dashboard.

> Output is for human review. The plate stage is a placeholder until you add a fine-tuned plate model (see README).

In [ ]:
import subprocess
print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout or "No GPU: it will run on CPU (slower).")

## 1. Get the code
Zip the project folder **without** `.venv`, `node_modules` and `models`, then upload it. (Or clone your repository instead.)

In [ ]:
import zipfile
from google.colab import files

# Option A: upload a zip of the project
uploaded = files.upload()
for name in uploaded:
    if name.endswith(".zip"):
        zipfile.ZipFile(name).extractall(".")
        print("extracted", name)

# Option B: clone your repository (uncomment and edit)
# !git clone https://github.com/<you>/traffic-violation-detection.git

In [ ]:
# Move into the project folder (the one that contains config.yaml)
import glob, os
cfgs = glob.glob("**/config.yaml", recursive=True)
assert cfgs, "config.yaml not found: upload the project zip first"
os.chdir(os.path.dirname(cfgs[0]) or ".")
print("working in", os.getcwd())

In [ ]:
!pip -q install -r requirements.txt

## 2. Upload a traffic video
Use a clip where the **stop line and the signal are both visible**. (Or mount Google Drive and set `VIDEO` to a path.)

In [ ]:
from google.colab import files
up = files.upload()
VIDEO = next(iter(up))
print("video:", VIDEO)

## 3. Define the stop line and signal ROI
Export a frame, look at it, note pixel coordinates, then pass them to the tools. `SIGNAL_FRAME` should show the light lit.

In [ ]:
CAMERA = "junction_1"
!python tools/define_line.py --video "$VIDEO" --export-frame frame.png
import matplotlib.pyplot as plt, matplotlib.image as mpimg
img = mpimg.imread("frame.png"); plt.figure(figsize=(12, 7)); plt.imshow(img); plt.grid(True, alpha=.3)
plt.title("Read pixel coordinates from the axes: stop-line ends and the signal box"); plt.show()

In [ ]:
# EDIT these from the frame above: stop line x1,y1,x2,y2 and signal box x,y,w,h (original pixels)
LINE = "120,540,880,560"
ROI = "812,96,38,110"
DIRECTION = "positive"   # use "negative" if the preview arrow points the wrong way
SIGNAL_FRAME = 0         # a frame where the light is clearly lit

!python tools/define_line.py --video "$VIDEO" --camera $CAMERA --points $LINE --direction $DIRECTION --preview line.png
!python tools/define_signal_roi.py --video "$VIDEO" --camera $CAMERA --roi $ROI --frame $SIGNAL_FRAME --scan 40 --preview roi.png
import matplotlib.pyplot as plt, matplotlib.image as mpimg
fig, ax = plt.subplots(1, 2, figsize=(16, 6)); ax[0].imshow(mpimg.imread("line.png")); ax[1].imshow(mpimg.imread("roi.png"))
[a.axis("off") for a in ax]; plt.show()

## 4. Run the pipeline

In [ ]:
!python main.py --video "$VIDEO" --config config.yaml --camera $CAMERA --model yolo26n.pt --tracker botsort --output outputs/annotated.mp4

In [ ]:
# Re-encode for in-browser playback and show it
!ffmpeg -y -loglevel error -i outputs/annotated.mp4 -vcodec libx264 -pix_fmt yuv420p outputs/annotated_h264.mp4
from IPython.display import HTML
from base64 import b64encode
data = b64encode(open("outputs/annotated_h264.mp4", "rb").read()).decode()
HTML(f'<video width="900" controls><source src="data:video/mp4;base64,{data}" type="video/mp4"></video>')

## 5. Events and evidence

In [ ]:
import pandas as pd
ev = pd.read_csv("outputs/events.csv", keep_default_na=False)
display(ev.event_type.value_counts().to_frame("count"))
display(ev.head(20))

In [ ]:
import glob, matplotlib.pyplot as plt, matplotlib.image as mpimg
folders = sorted(glob.glob("evidence/*"))[:3]
for f in folders:
    names = ["before", "crossing", "after", "vehicle", "plate"]
    fig, ax = plt.subplots(1, len(names), figsize=(20, 3.5)); fig.suptitle(f)
    for a, n in zip(ax, names):
        try:
            a.imshow(mpimg.imread(f"{f}/{n}.jpg"))
        except FileNotFoundError:
            a.text(.5, .5, "n/a", ha="center")
        a.set_title(n); a.axis("off")
    plt.show()
if not folders:
    print("No violations were flagged. Check the line, direction and signal ROI above.")

## 6. Evaluate and benchmark (optional)
For evaluation, create `ground_truth/ground_truth.csv` (see `ground_truth/ground_truth_template.csv`).

In [ ]:
import os
if os.path.exists("ground_truth/ground_truth.csv"):
    !python evaluate.py --pred outputs/events.csv --gt ground_truth/ground_truth.csv --tolerance 15
else:
    print("Add ground_truth/ground_truth.csv first.")

In [ ]:
!python benchmark.py --video "$VIDEO" --camera $CAMERA --models yolov8n.pt yolo26n.pt --trackers botsort bytetrack --max-frames 300

## 7. Dashboard (optional)
Builds the web UI, starts the API and opens a public Cloudflare quick-tunnel. **A token is generated**, because the dashboard shows plates. The URL stops working when this session ends.

In [ ]:
!curl -fsSL https://deb.nodesource.com/setup_20.x | bash - >/dev/null 2>&1 && apt-get install -y nodejs >/dev/null 2>&1
!cd dashboard/web && npm ci --no-audit --no-fund && npm run build

In [ ]:
import os, secrets, subprocess, time
TOKEN = secrets.token_urlsafe(24)
os.environ["TVD_TOKEN"] = TOKEN
api = subprocess.Popen(["uvicorn", "dashboard.api.main:app", "--host", "0.0.0.0", "--port", "8000"])
time.sleep(4)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
print("Access token (paste it in the dashboard):", TOKEN)
tunnel = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8000"], stderr=subprocess.PIPE, text=True)
for line in tunnel.stderr:
    if "trycloudflare.com" in line:
        print(line.strip())
        break

In [ ]:
# Stop the dashboard and tunnel when you are done
tunnel.terminate()
api.terminate()